# Optimizer comparison on classification using Digits dataset and Logistic Regression

In [1]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import Logistic
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-logreg-digits"

In [2]:
train_loader, test_loader = data_load_and_prep("digits", test_size=0.3, random_state=42, batch_size='full')

## Adam

In [3]:
loss_fn = nn.CrossEntropyLoss()
adam_model = Logistic(input_dim=64, output_dim=10)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Adam optimizaer

In [4]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=50, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=2.1357 | train_acc=0.238 | test_loss=2.1387 | test_acc=0.239 | 
Epoch 005 | train_loss=1.2248 | train_acc=0.749 | test_loss=1.2470 | test_acc=0.748 | 
Epoch 010 | train_loss=0.7640 | train_acc=0.847 | test_loss=0.7863 | test_acc=0.848 | 
Epoch 015 | train_loss=0.5386 | train_acc=0.882 | test_loss=0.5618 | test_acc=0.867 | 
Epoch 020 | train_loss=0.4156 | train_acc=0.905 | test_loss=0.4407 | test_acc=0.878 | 
Epoch 025 | train_loss=0.3403 | train_acc=0.922 | test_loss=0.3672 | test_acc=0.900 | 
Epoch 030 | train_loss=0.2906 | train_acc=0.942 | test_loss=0.3186 | test_acc=0.911 | 
Epoch 035 | train_loss=0.2559 | train_acc=0.951 | test_loss=0.2848 | test_acc=0.920 | 


2026/03/30 14:45:28 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 040 | train_loss=0.2302 | train_acc=0.955 | test_loss=0.2610 | test_acc=0.930 | 
Epoch 045 | train_loss=0.2103 | train_acc=0.959 | test_loss=0.2440 | test_acc=0.939 | 
Epoch 049 | train_loss=0.1973 | train_acc=0.961 | test_loss=0.2338 | test_acc=0.943 | 


## l-BFGS optimizer

In [5]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = Logistic(64, 10)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [6]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=2.5632 | train_acc=0.048 | test_loss=2.5660 | test_acc=0.074 | 
Epoch 005 | train_loss=2.2644 | train_acc=0.189 | test_loss=2.2707 | test_acc=0.178 | 
Epoch 010 | train_loss=2.0026 | train_acc=0.367 | test_loss=2.0124 | test_acc=0.343 | 
Epoch 015 | train_loss=1.7965 | train_acc=0.523 | test_loss=1.8095 | test_acc=0.528 | 
Epoch 020 | train_loss=1.6322 | train_acc=0.670 | test_loss=1.6470 | test_acc=0.633 | 
Epoch 025 | train_loss=1.4994 | train_acc=0.765 | test_loss=1.5146 | test_acc=0.720 | 
Epoch 030 | train_loss=1.3851 | train_acc=0.819 | test_loss=1.4001 | test_acc=0.778 | 
Epoch 035 | train_loss=1.2859 | train_acc=0.859 | test_loss=1.3010 | test_acc=0.837 | 
Epoch 040 | train_loss=1.1987 | train_acc=0.882 | test_loss=1.2140 | test_acc=0.867 | 
Epoch 045 | train_loss=1.1213 | train_acc=0.902 | test_loss=1.1369 | test_acc=0.887 | 
Epoch 049 | train_loss=1.0651 | train_acc=0.913 | test_loss=1.0811 | test_acc=0.898 | 


2026/03/30 14:45:32 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


## Newton-Raphson optimizer

In [7]:
loss_fn = nn.CrossEntropyLoss()
newton_model = Logistic(64, 10)
classical_device = "cuda" 
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                              lr=0.1,
                              max_iter=1,
                              damping=1e-4,
                              )

In [8]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.7738 | train_acc=0.489 | test_loss=1.7876 | test_acc=0.469 | 
Epoch 005 | train_loss=1.1387 | train_acc=0.954 | test_loss=0.8512 | test_acc=0.926 | 
Epoch 010 | train_loss=0.4920 | train_acc=0.975 | test_loss=0.5242 | test_acc=0.943 | 
Epoch 015 | train_loss=0.3131 | train_acc=0.986 | test_loss=0.3568 | test_acc=0.957 | 
Epoch 020 | train_loss=0.2047 | train_acc=0.995 | test_loss=0.2611 | test_acc=0.957 | 
Epoch 025 | train_loss=0.1351 | train_acc=0.998 | test_loss=0.2041 | test_acc=0.956 | 
Epoch 030 | train_loss=0.0892 | train_acc=0.999 | test_loss=0.1705 | test_acc=0.961 | 
Epoch 035 | train_loss=0.0587 | train_acc=1.000 | test_loss=0.1519 | test_acc=0.963 | 
Epoch 040 | train_loss=0.0385 | train_acc=1.000 | test_loss=0.1429 | test_acc=0.965 | 
Epoch 045 | train_loss=0.0252 | train_acc=1.000 | test_loss=0.1398 | test_acc=0.967 | 


2026/03/30 14:45:47 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.0181 | train_acc=1.000 | test_loss=0.1399 | test_acc=0.967 | 


## Quadriatic Annealing optimizer (cpu)

In [9]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(64, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [10]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-optimizer"
)

Epoch 000 | train_loss=2.5095 | train_acc=0.060 | test_loss=2.4867 | test_acc=0.072 | 
Epoch 005 | train_loss=2.0950 | train_acc=0.278 | test_loss=2.0719 | test_acc=0.291 | 
Epoch 010 | train_loss=1.7651 | train_acc=0.476 | test_loss=1.7308 | test_acc=0.489 | 
Epoch 015 | train_loss=1.4985 | train_acc=0.603 | test_loss=1.4522 | test_acc=0.631 | 
Epoch 020 | train_loss=1.2799 | train_acc=0.698 | test_loss=1.2272 | test_acc=0.707 | 
Epoch 025 | train_loss=1.0944 | train_acc=0.755 | test_loss=1.0461 | test_acc=0.763 | 
Epoch 030 | train_loss=0.9373 | train_acc=0.815 | test_loss=0.8930 | test_acc=0.815 | 
Epoch 035 | train_loss=0.8067 | train_acc=0.848 | test_loss=0.7717 | test_acc=0.856 | 
Epoch 040 | train_loss=0.6971 | train_acc=0.877 | test_loss=0.6650 | test_acc=0.878 | 


2026/03/30 14:45:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 045 | train_loss=0.6063 | train_acc=0.895 | test_loss=0.5829 | test_acc=0.894 | 
Epoch 049 | train_loss=0.5444 | train_acc=0.897 | test_loss=0.5230 | test_acc=0.887 | 


## Quadratic Annealing optimizer (cpu + momentum)

In [11]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(64, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.1,
    num_reads=100,
    beta=0.9,
)

In [12]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-momentum-optimizer"
)

Epoch 000 | train_loss=2.1481 | train_acc=0.232 | test_loss=2.1472 | test_acc=0.215 | 
Epoch 005 | train_loss=1.2371 | train_acc=0.694 | test_loss=1.2133 | test_acc=0.676 | 
Epoch 010 | train_loss=0.7025 | train_acc=0.849 | test_loss=0.6848 | test_acc=0.850 | 
Epoch 015 | train_loss=0.4432 | train_acc=0.893 | test_loss=0.4429 | test_acc=0.878 | 
Epoch 020 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
Epoch 025 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
Epoch 030 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
Epoch 035 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
Epoch 040 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
Epoch 045 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 


2026/03/30 14:45:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=0.2928 | train_acc=0.930 | test_loss=0.3006 | test_acc=0.922 | 
